In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.inspection import permutation_importance

# 1. Load Data
selected_features_df = pd.read_csv('../../data/03_primary/selected_features.csv')
churn_data = pd.read_csv('../../data/03_primary/churn_features.csv')

# 2. Prepare Features (X) and Target (y)
# Extract the list of features we want to keep
selected_features = selected_features_df['feature'].tolist()
target_col = 'customer_churn'

# Filter the dataframe to only include selected features + target
valid_features = [f for f in selected_features if f in churn_data.columns]
X = churn_data[valid_features].copy()
y = churn_data[target_col]

# Handle Missing Values 
X = X.fillna(0)

# 3. Train/Test Split (80/20)
# stratify=y ensures we keep the same ratio of churners in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Handle Class Imbalance
# Heavily penalizes the model if it misses an actual churner
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

# 5. Train Gradient Boosting Model
print("Training Gradient Boosting Classifier...")
model = HistGradientBoostingClassifier(
    max_iter=150,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)
model.fit(X_train, y_train, sample_weight=sample_weights)

# 6. Evaluate Model
y_pred = model.predict(X_test)

print("\n" + "="*40)
print("MODEL EVALUATION")
print("="*40)
print("Classification Report:\n", classification_report(y_test, y_pred))

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# 7. Feature Importance Calculation
print("\nCalculating Feature Importances...")
result = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': result.importances_mean
}).sort_values(by='Importance', ascending=False)

print("\nTop 10 Important Features:")
print(importance_df.head(10))

Training Gradient Boosting Classifier...

MODEL EVALUATION
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.95      0.97      3329
           1       0.84      0.95      0.89       977

    accuracy                           0.95      4306
   macro avg       0.91      0.95      0.93      4306
weighted avg       0.95      0.95      0.95      4306

Confusion Matrix:
 [[3149  180]
 [  44  933]]

Calculating Feature Importances...

Top 10 Important Features:
                   Feature  Importance
2                total_van    0.126846
0         avg_days_to_pull    0.123409
3      avg_resolution_days    0.122341
1                  avg_van    0.106642
14    company_size_encoded    0.007617
16         any_van_changed    0.004041
5          total_bob_value    0.004041
4             num_machines    0.002369
9   avg_agreement_duration    0.002090
12           num_contracts    0.000790
